In [ ]:
import numpy as np
from sklearn.svm import SVR



# ---------------------
# SVR surrogate
# ---------------------
# Polynomial kernel for better extrapolation
svr_model = SVR(kernel='poly', degree=3, C=100, gamma=1, coef0=1, epsilon=1e-3)
svr_model.fit(X_train, y_train)

# ---------------------
# Candidate sampling
# ---------------------
def sample_candidates(n_samples=1000, input_dim=6):
    # Adjust ranges if your input features are not [0,1]
    return np.random.rand(n_samples, input_dim)

# ---------------------
# Acquisition function
# ---------------------
# Simple: maximize predicted output
def acquisition(candidates):
    preds = svr_model.predict(candidates)
    return preds

# ---------------------
# Bayesian Optimization loop
# ---------------------
num_iterations = 80
input_dim = X_train.shape[1]

for i in range(num_iterations):
    # Sample candidate points
    candidates = sample_candidates(n_samples=1000, input_dim=input_dim)

    # Predict outputs
    predicted_outputs = acquisition(candidates)

    # Choose the candidate with the highest predicted output
    idx_next = np.argmax(predicted_outputs)
    next_input = candidates[idx_next]
    next_predicted_output = predicted_outputs[idx_next]

    print(f"Iteration {i+1}: Next input = {next_input}, Predicted output = {next_predicted_output:.4f}")

    # Simulate adding this new point to training data
    # In practice, you would evaluate your real system here
    X_train = np.vstack([X_train, next_input])
    y_train = np.append(y_train, next_predicted_output)

    # Refit SVR with updated data
    svr_model.fit(X_train, y_train)


In [ ]:
from sklearn.svm import SVR
import numpy as np
#  week 4 approach
X_train_copy = X_train
y_train_copy = y_train


# Initialize SVR
svr_model = SVR(kernel='rbf')  # can try 'poly' later if desired
svr_model.fit(X_train_copy, y_train_copy)

input_dim = X_train.shape[1]
num_iterations = 30
num_candidates = 100

# Lists to store candidates and predicted outputs
all_inputs = []
all_outputs = []

for i in range(num_iterations):
    # Sample candidate inputs randomly
    candidates = np.random.rand(num_candidates, input_dim)

    # Predict outputs using current SVR
    preds = svr_model.predict(candidates)

    # Store all
    all_inputs.extend(candidates)
    all_outputs.extend(preds)

    # Pick the candidate with the highest predicted output
    idx_next = np.argmax(preds)
    next_input = candidates[idx_next]
    next_output = preds[idx_next]

    print(f"Iteration {i+1}: Next input = {next_input}, Predicted output = {next_output:.4f}")

    # Optional: add this best candidate to training set to refine SVR
    X_train = np.vstack([X_train_copy, next_input])
    y_train = np.append(y_train_copy, next_output)
    svr_model.fit(X_train_copy, y_train_copy)  # retrain SVR

# After loop, find overall maximum
all_inputs = np.array(all_inputs)
all_outputs = np.array(all_outputs)
max_idx = np.argmax(all_outputs)

print("\nHighest predicted output overall:")
print(f"Input: {all_inputs[max_idx]}")
print(f"Output: {all_outputs[max_idx]:.4f}")
